# 03 Attention network

In this notebook, we build the main model from the plan (`01-Research-and-Plan.md`): a network that reads each
report as a bag of FinBERT chunk vectors, learns a weight for each chunk, and predicts what the straight line gets
wrong. The variants are:

| Variant | What changes | Why |
|---|---|---|
| Attention, Item 7 | Nothing, this is the main model | The section Kogan et al. used |
| Attention, Item 1 | The business description instead | Which section carries the risk information? |
| Attention, both | Both sections in one bag, with a learned marker saying which section a chunk is from | Does the model use both, and which does it weight? |
| Mean pooling, Item 7 | The same network with every chunk weighted equally | Isolates what learning the weights buys |

The bar is the within-year ridge from `02-Linear-models.ipynb`, on the same inputs. Everything follows Oscar's setup
in `report/02-Data.ipynb`: `data.pkl`, test years 2001 to 2006 with the paper and strict windows, and the last
training year used for early stopping. Every variant is scored with `scoring.py` against the straight line. Where
the course covers a method we cite the lecture and slide, and go to papers only for what it does not.

**On a Colab GPU this notebook trains everything (section 6); anywhere else it only builds and checks the model.**
Section 5 pushes a batch through the model and checks each piece, and estimates how long the full run would take on
the machine it is running on.

On Colab, the first cell connects Google Drive, gets the repo and copies `data.pkl`. Locally it does nothing. The
full run needs a GPU (Runtime, Change runtime type, T4); the checks run anywhere.

In [ ]:
import importlib.util, os, shutil, subprocess
from pathlib import Path

NOTEBOOK_DIR = "Alex"
IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    repo = Path("/content/dst-sprint2")
    if repo.exists():
        subprocess.run(["git", "-C", str(repo), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/Oscar339/dst-sprint2.git", str(repo)], check=True)
    processed = repo / "data" / "processed"
    processed.mkdir(parents=True, exist_ok=True)
    if not (processed / "data.pkl").exists():
        shutil.copy(Path("/content/drive/MyDrive/sprint2-data/data.pkl"), processed / "data.pkl")
    os.chdir(repo / NOTEBOOK_DIR)

In [ ]:
import copy, pickle, sys, time, platform
sys.path.insert(0, ".")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn

import scoring

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Python {platform.python_version()} | torch {torch.__version__} | device {DEVICE}"
      + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else " (checks only; train on a Colab GPU)"))

with open("../data/processed/data.pkl", "rb") as f:   # Oscar's hand-over file (Oscar/make_handover.py)
    d = pickle.load(f)
df = scoring.frame(d)
CHUNKS = {"Item 1": d["X_item1"], "Item 7": d["X_item7"]}        # per report: an (n_chunks x 768) float16 array
x_all, y_all, yr_all = df["logvol_before"].to_numpy(), df["logvol_after"].to_numpy(), df["year"].to_numpy()
n_chunks = {s: np.array([len(c) for c in CHUNKS[s]]) for s in CHUNKS}
print(f"{len(df):,} reports; chunks per report: Item 1 median {np.median(n_chunks['Item 1']):.0f}, "
      f"Item 7 median {np.median(n_chunks['Item 7']):.0f}, max {max(n_chunks['Item 1'].max(), n_chunks['Item 7'].max())} per section")

## 1. The model

A report is a bag of chunk vectors $\mathbf{c}_1, \dots, \mathbf{c}_n$ (768 numbers each, up to 64 per section).
Following attention-based multiple-instance learning (Ilse, Tomczak & Welling, 2018), the network scores each chunk,
turns the scores into weights that sum to one, and averages the chunks with those weights:

$$\mathbf{h}_i = \mathrm{ReLU}(W\mathbf{c}_i + \mathbf{b}) + \mathbf{s}_{\text{section}(i)}, \qquad
a_i = \frac{\exp\big(\mathbf{w}^\top \tanh(V\mathbf{h}_i)\big)}{\sum_j \exp\big(\mathbf{w}^\top \tanh(V\mathbf{h}_j)\big)}, \qquad
\mathbf{z} = \sum_i a_i \mathbf{h}_i .$$

* Projection - $W$ maps the 768 FinBERT numbers to a smaller vector (128). The 768 numbers are heavily
  correlated, and with about 10,000 training reports a smaller vector keeps the parameter count down (about 115,000).
* Section marker - $\mathbf{s}$ is a learned vector per section, added to every chunk of that section, so the
  "both" variant can tell Item 1 from Item 7. With one section it is just a shared offset.
* Attention - the weights $a_i$ are a softmax over the chunks, with padding excluded. They depend only on each
  chunk's own content, so shuffling the chunks does not change the answer: the report is treated as a bag, not a
  sequence. This is the difference from FETILDA (Xia et al., 2022), which runs a bi-LSTM over the chunks in order.
  The weights are also what we plot afterwards to see which parts of a report the model relied on.
* Head - a small regression head $\hat r = \mathbf{w}_2^\top\mathrm{ReLU}(W_2\mathbf{z} + \mathbf{b}_2) + b_3$.

Where this comes from in the course: the weights $a_i$ are a softmax (lecture 06.1, slide 7) over scores from a
one-hidden-layer network with a tanh activation (slide 10); the projection and the head are the multilayer
perceptron of slide 12, with ReLU; and the weighted sum is a pooling layer (lecture 06.2, slide 12) with learned
rather than fixed weights, so the mean-pooling variant is that lecture's average pooling applied to chunks.
Attention itself is only a discussion question in lecture 06.2 (slide 17), so its form here is from Ilse et al.
(2018). Lecture 06.1 (slide 13) notes that a wide enough one-hidden-layer network can fit anything and asks why you
should not build one: with about 10,000 training reports we keep the network small and regularise it (section 3).

The network predicts the **within-year residual** of the straight line, as ridge did, and the prediction is
$\hat y = a + b \cdot \text{logvol\_before} + \hat r$. Setting every $a_i = 1/n$ gives the mean-pooling variant, so
the comparison between the two is exactly what learning the weights buys.

In [ ]:
class AttentionNet(nn.Module):
    def __init__(self, width=128, attention=64, head=64, dropout=0.2, pooling="attention"):
        super().__init__()
        self.pooling = pooling
        self.project = nn.Sequential(nn.Linear(768, width), nn.ReLU(), nn.Dropout(dropout))
        self.section = nn.Embedding(2, width)
        self.attn_V = nn.Linear(width, attention)
        self.attn_w = nn.Linear(attention, 1, bias=False)
        self.head = nn.Sequential(nn.Linear(width, head), nn.ReLU(), nn.Dropout(dropout), nn.Linear(head, 1))

    def forward(self, x, section, mask):
        # x (B, L, 768) standardised chunk vectors; section (B, L) 0/1; mask (B, L) True where there is a chunk
        h = self.project(x) + self.section(section)
        if self.pooling == "attention":
            score = self.attn_w(torch.tanh(self.attn_V(h))).squeeze(-1)
        else:                                                       # mean pooling: every chunk scores the same
            score = torch.zeros(h.shape[:2], device=h.device)
        score = score.masked_fill(~mask, float("-inf"))
        a = torch.softmax(score, dim=1)                             # (B, L), sums to 1 over the real chunks
        z = (a.unsqueeze(-1) * h).sum(1)
        return self.head(z).squeeze(-1), a

## 2. From chunk lists to batches

`data.pkl` stores each report's chunks as a separate array, as reports have different numbers of chunks. A batch
pads its reports to the longest one in the batch and carries a mask saying which positions are real. The chunk
vectors are standardised per dimension with the mean and standard deviation of the training reports' chunks, as the
ridge inputs were. Padding every report in a batch to the same length turns a loop over reports into one tensor
operation: the vectorisation of lecture 09.1 (slides 11 to 13), and what lets a GPU, with its many simple cores built
for matrix operations (slide 5), run it.

In [ ]:
SECTION_ID = {"Item 1": 0, "Item 7": 1}

def sections_of(label):
    return ["Item 1", "Item 7"] if label == "both" else [label]

def chunk_stats(idx, sections):
    # Mean and sd per dimension over every chunk of the reports in idx (running sums, so nothing big is built)
    s, ss, n = np.zeros(768), np.zeros(768), 0
    for sec in sections:
        for i in idx:
            c = CHUNKS[sec][i].astype(np.float32)
            s += c.sum(0); ss += (c * c).sum(0); n += len(c)
    mu = s / n
    return mu.astype(np.float32), np.sqrt(np.maximum(ss / n - mu * mu, 1e-8)).astype(np.float32)

def make_batch(idx, sections, stats):
    mu, sd = stats
    seqs = [np.concatenate([CHUNKS[sec][i] for sec in sections]).astype(np.float32) for i in idx]
    secs = [np.concatenate([np.full(len(CHUNKS[sec][i]), SECTION_ID[sec]) for sec in sections]) for i in idx]
    L = max(len(q) for q in seqs)
    x = np.zeros((len(idx), L, 768), np.float32)
    section = np.zeros((len(idx), L), np.int64)
    mask = np.zeros((len(idx), L), bool)
    for k, (q, sc) in enumerate(zip(seqs, secs)):
        x[k, :len(q)] = (q - mu) / sd
        section[k, :len(q)] = sc
        mask[k, :len(q)] = True
    return (torch.from_numpy(x).to(DEVICE), torch.from_numpy(section).to(DEVICE), torch.from_numpy(mask).to(DEVICE))

## 3. Training and early stopping

The training loop is the one from workshop 06.3: for each batch, a forward pass, the loss, `zero_grad`, `backward`
and `step`, with `model.eval()` when scoring. The loss is MSE (lecture 06.1, slide 15) on the target below, and the
weights move down the gradient by a step set by the learning rate (slide 16). Two things differ from the workshop,
which trains for a fixed ten epochs and checks the *test* set after every one:

* Nothing is tuned on the test year (Oscar's rule, and lecture 02.2 slide 7): the last training year is the
  validation year. For a window of years $t_1 \dots t_k$, the network is fitted on $t_1 \dots t_{k-1}$, scored on
  $t_k$ after every epoch, and the epoch with the lowest validation MSE is kept. Training stops when the validation
  year has not improved for `patience` epochs. This is one year less than ridge had, which refitted on the whole
  window with the chosen $\lambda$; we note it and keep the comparison with ridge on the same test reports.
* Three things the course does not cover, taken from the literature: *early stopping*, above, the usual way to
  regularise a network (Hastie, Tibshirani & Friedman, 2009, §11.5); *dropout*, which switches off a random
  fraction of the hidden units at each training step so no single unit can be relied on (Srivastava et al.,
  2014); and *AdamW* in place of plain gradient descent, which scales each weight's step by the recent size of its
  gradients and applies weight decay, a penalty like ridge's, separately (Kingma & Ba, 2015; Loshchilov & Hutter,
  2019). Louis's network uses the same optimiser, so the two networks are comparable.

The target is the straight line's residual with each training year's average taken off (the within-year target of
`02-Linear-models.ipynb`), divided by its standard deviation so the loss is on a sensible scale.

| Setting | Value | Why |
|---|---|---|
| `width` | 128 | Projection of the 768 FinBERT numbers; keeps the parameter count near 115,000 |
| `attention` | 64 | Hidden size of the attention scorer $V$; Ilse et al. used 128 for larger bags |
| `head` | 64 | One hidden layer in the regression head |
| `dropout` | 0.2 | After the projection and in the head |
| `lr`, `weight_decay` | 0.001, 0.01 | AdamW; a standard starting point, the same as Louis's network |
| `batch` | 64 reports | About 2,000 to 4,000 chunks per batch |
| `max_epochs`, `patience` | 30, 5 | Early stopping on the validation year |
| `seeds` | 0 to 4 | The spread between seeds measures training noise |

These are starting values. If the validation curves show the model is still improving at 30 epochs, or stops in
the first two or three, `max_epochs` and `lr` are the first things to change.

In [ ]:
HP = dict(width=128, attention=64, head=64, dropout=0.2, lr=1e-3, weight_decay=1e-2,
          batch=64, max_epochs=30, patience=5, seeds=[0, 1, 2, 3, 4])
INPUTS = ["Item 7", "Item 1", "both"]
RUN_FULL = IN_COLAB and DEVICE == "cuda"     # the full run (section 6) happens only on a Colab GPU

# Outputs go to Google Drive on Colab, so they survive the session ending; the last cell copies them into the repo.
OUT_DIR = Path("/content/drive/MyDrive/sprint2-data/attention") if IN_COLAB else Path("../data/processed/attention")
CHECKPOINT_DIR = OUT_DIR / "checkpoints"
RESULTS_FILE = OUT_DIR / "alex-03-attention-results.csv"
RUNS_FILE = OUT_DIR / "alex-03-attention-runs.csv"

def batches(idx, size, rng=None):
    order = rng.permutation(idx) if rng is not None else np.asarray(idx)
    for i in range(0, len(order), size):
        yield order[i:i + size]

def predict_text(model, idx, sections, stats, hp=HP):
    # The network's part of the prediction, in standardised residual units
    model.eval()
    out = []
    with torch.no_grad():
        for b in batches(idx, 4 * hp["batch"]):
            out.append(model(*make_batch(b, sections, stats))[0].cpu().numpy())
    return np.concatenate(out)

def train_one(fit_idx, val_idx, sections, target, stats, seed, hp=HP, pooling="attention", log=print):
    # Fit on fit_idx, keep the epoch with the best MSE on val_idx. Returns the model and its history.
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = AttentionNet(hp["width"], hp["attention"], hp["head"], hp["dropout"], pooling).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])
    loss_fn = nn.MSELoss()
    t_all = torch.from_numpy(target.astype(np.float32)).to(DEVICE)
    best, wait, history = (np.inf, None, -1), 0, []
    for epoch in range(hp["max_epochs"]):
        model.train()
        t0, total = time.perf_counter(), 0.0
        for b in batches(fit_idx, hp["batch"], rng):
            opt.zero_grad()
            pred, _ = model(*make_batch(b, sections, stats))
            loss = loss_fn(pred, t_all[b])
            loss.backward()
            opt.step()
            total += loss.item() * len(b)
        seconds = time.perf_counter() - t0
        val_mse = float(np.mean((predict_text(model, val_idx, sections, stats, hp) - target[val_idx]) ** 2))
        history.append({"epoch": epoch, "train_mse": total / len(fit_idx), "val_mse": val_mse, "seconds": seconds})
        if val_mse < best[0]:
            best, wait = (val_mse, copy.deepcopy(model.state_dict()), epoch), 0
        else:
            wait += 1
            if wait >= hp["patience"]:
                break
    model.load_state_dict(best[1])
    if log:
        log(f"  seed {seed}: stopped after {len(history)} epochs, best epoch {best[2]}, "
            f"val MSE {best[0]:.3f} (standardised), {np.mean([h['seconds'] for h in history]):.1f}s/epoch")
    return model, pd.DataFrame(history)

## 4. Scoring

`scoring.evaluate` calls a model as `predict(df, train, test)` for every test year and window, and compares it with
the straight line by a paired bootstrap over the test reports (lecture 05.2, slide 14). The wrapper below
does the whole thing for one window: fits the straight line on the training reports, builds the within-year target,
splits off the validation year, trains the network, and returns line + network for the test reports. The network's
part is centred on the test year's own average, so, as for the within-year ridge, it can only change how companies
are ranked and never the year's level. That uses the other test reports' text, never their volatility.

A log of every run (epochs, time, best validation MSE) is kept for the scaling section.

In [ ]:
RUN_LOG = []

def make_attention(label, seed, pooling="attention", hp=HP):
    sections = sections_of(label)
    def predict(df, train, test):
        years = sorted(set(yr_all[train]))
        fit, val = train & (yr_all != years[-1]), train & (yr_all == years[-1])
        fit_idx, val_idx, test_idx = np.flatnonzero(fit), np.flatnonzero(val), np.flatnonzero(test)
        a, b = scoring.fit_line(df, train)
        resid = y_all - (a + b * x_all)
        resid = resid - pd.Series(resid).groupby(yr_all).transform("mean").where(train, 0).to_numpy()   # demean the training years
        scale = resid[fit_idx].std()
        stats = chunk_stats(fit_idx, sections)
        t0 = time.perf_counter()
        model, history = train_one(fit_idx, val_idx, sections, resid / scale, stats, seed, hp, pooling)
        text = predict_text(model, test_idx, sections, stats, hp) * scale
        text = text - text.mean()                                                       # centre on the test year
        RUN_LOG.append({"input": label, "pooling": pooling, "seed": seed, "test_year": int(yr_all[test_idx[0]]),
                        "train_years": f"{years[0]}-{years[-1]}", "fit_reports": len(fit_idx), "epochs": len(history),
                        "best_epoch": int(history["val_mse"].idxmin()), "best_val_mse": float(history["val_mse"].min() * scale ** 2),
                        "seconds": time.perf_counter() - t0, "seconds_per_epoch": float(history["seconds"].mean())})
        if CHECKPOINT_DIR.exists():
            torch.save(model.state_dict(), CHECKPOINT_DIR / f"{label.replace(' ', '')}_{pooling}_{years[0]}-{years[-1]}_seed{seed}.pt")
        return a + b * x_all[test_idx] + text
    return predict

## 5. Checks before training

Before any real training, we check each piece on a handful of reports:

1. the parameter count, and that a batch goes through with the right shapes;
2. the attention weights sum to one over the real chunks and are exactly zero on the padding;
3. shuffling a report's chunks does not change its prediction (a bag, not a sequence), and the mean-pooling variant
   gives every chunk the same weight;
4. the network can overfit a few reports, so the gradients flow through the attention;
5. the throughput on this machine, and from it an estimate of the full run.

In [ ]:
model = AttentionNet(HP["width"], HP["attention"], HP["head"], HP["dropout"]).to(DEVICE).eval()
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

idx = np.arange(8)
stats = chunk_stats(np.arange(2000), ["Item 1", "Item 7"])
x, section, mask = make_batch(idx, ["Item 1", "Item 7"], stats)
with torch.no_grad():
    out, a = model(x, section, mask)
print(f"batch of 8, both sections: x {tuple(x.shape)}, output {tuple(out.shape)}, weights {tuple(a.shape)}")
print(f"real chunks per report: {mask.sum(1).tolist()}")
assert torch.allclose(a.sum(1), torch.ones(8, device=DEVICE), atol=1e-5), "weights must sum to 1"
assert (a[~mask] == 0).all(), "padding must get zero weight"
print("weights sum to 1 over the real chunks, and are 0 on the padding")

# 3. a bag, not a sequence
perm = torch.randperm(int(mask[0].sum()), device=DEVICE)
x2, s2 = x[:1].clone(), section[:1].clone()
x2[0, :len(perm)], s2[0, :len(perm)] = x[0, perm], section[0, perm]
with torch.no_grad():
    out2, _ = model(x2, s2, mask[:1])
    assert torch.allclose(out[:1], out2, atol=1e-5), "shuffling the chunks changed the prediction"
    _, a_mean = AttentionNet(pooling="mean").to(DEVICE).eval()(x, section, mask)
assert torch.allclose(a_mean[mask], (1 / mask.sum(1, keepdim=True).expand_as(a_mean))[mask].float(), atol=1e-6)
print("shuffling the chunks leaves the prediction unchanged; mean pooling gives equal weights")

In [ ]:
# 4. Can it overfit 32 reports? The loss should fall well below where it started.
small = np.arange(32)
train, test = scoring.masks(df, 2006, "strict")
a_line, b_line = scoring.fit_line(df, train)
resid = (y_all - (a_line + b_line * x_all)); resid = resid / resid[train].std()
hp_small = dict(HP, batch=32, max_epochs=40, patience=40)
t0 = time.perf_counter()
m, hist = train_one(small, small, ["Item 7"], resid, stats, seed=0, hp=hp_small, log=None)
print(f"overfit test: training MSE epoch 1 {hist.train_mse.iloc[0]:.3f} -> epoch 40 {hist.train_mse.iloc[-1]:.3f} "
      f"({time.perf_counter() - t0:.0f}s)")
assert hist.train_mse.iloc[-1] < 0.5 * hist.train_mse.iloc[0], "the network did not learn the 32 reports"

In [ ]:
# 5. Throughput on this machine, and what the full run would take
sample = np.flatnonzero(train)[:512]
m = AttentionNet(HP["width"], HP["attention"], HP["head"], HP["dropout"]).to(DEVICE)
opt = torch.optim.AdamW(m.parameters(), lr=HP["lr"])
t_all = torch.from_numpy(resid.astype(np.float32)).to(DEVICE)
t0 = time.perf_counter()
for b in batches(sample, HP["batch"]):
    opt.zero_grad(); loss = nn.MSELoss()(m(*make_batch(b, ["Item 7"], stats))[0], t_all[b]); loss.backward(); opt.step()
if DEVICE == "cuda":
    torch.cuda.synchronize()
per_second = len(sample) / (time.perf_counter() - t0)

runs = len(INPUTS) * len(scoring.WINDOWS) * len(scoring.TEST_YEARS) * len(HP["seeds"])
mean_fit_reports = np.mean([ (scoring.masks(df, T, w)[0] & (yr_all != max(scoring.WINDOWS[w](T)))).sum()
                             for w in scoring.WINDOWS for T in scoring.TEST_YEARS])
epochs_guess = 15
hours = runs * mean_fit_reports * epochs_guess / per_second / 3600
print(f"training throughput on {DEVICE}: {per_second:,.0f} reports/s (Item 7, batch {HP['batch']})")
print(f"full run: {runs} training runs x about {mean_fit_reports:,.0f} reports x ~{epochs_guess} epochs "
      f"= about {hours:.1f} hours on this machine (plus the mean-pooling variant, 1/3 more)")

## 6. The full run

Runs only on a Colab GPU (`RUN_FULL`). Every variant is scored on both windows and all six test years with five
seeds: 240 training runs. The results and a log of every run are appended to Google Drive
(`sprint2-data/attention/`) after each variant and seed, and a rerun skips what is already there, so a session that
stops part-way is restarted from where it got to. Seeds are the outer loop, so if time runs out every variant has
the same number of seeds. Trained weights go to the same folder. The last step copies the two result files into
`data/output/`, to be committed.

The 240 training runs are independent of each other - embarrassingly parallel in the sense of lecture 09.1 (slides
9 to 10) - so on a cluster they would all run at once. Within a run the epochs are sequential, which is the part
that Amdahl's law (lecture 09.2, slide 6) says no amount of hardware removes. Here they run one after another on one
GPU.

In [ ]:
if RUN_FULL:
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    variants = [(label, "attention") for label in INPUTS] + [("Item 7", "mean")]
    done = pd.read_csv(RESULTS_FILE)["model"].unique().tolist() if RESULTS_FILE.exists() else []
    t_start = time.perf_counter()
    for seed in HP["seeds"]:                        # seeds outermost: if the session dies, every variant has the same seeds
        for label, pooling in variants:
            name = f"{pooling} pooling, {label}, seed {seed}"
            if name in done:
                continue
            print(f"{name}   ({(time.perf_counter() - t_start) / 60:.0f} min so far)")
            res = scoring.evaluate(df, make_attention(label, seed, pooling), name=name)
            res.to_csv(RESULTS_FILE, mode="a", header=not RESULTS_FILE.exists(), index=False)
            pd.DataFrame(RUN_LOG).to_csv(RUNS_FILE, mode="a", header=not RUNS_FILE.exists(), index=False)
            RUN_LOG.clear()
    for f in [RESULTS_FILE, RUNS_FILE]:             # into the repo, to commit from
        shutil.copy(f, Path("../data/output") / f.name)
    print(f"done: {pd.read_csv(RESULTS_FILE)['model'].nunique()} variant-seed runs in {RESULTS_FILE}, "
          f"copied to data/output/")
else:
    print("RUN_FULL is False: nothing trained. On a Colab GPU it is True and this cell runs everything.")

## 7. What this shows

To be written after the full run. The things to look at:

* Per seed and averaged over seeds, the change in MSE against the straight line, and against the within-year ridge
  on the same inputs - does learning the weights beat averaging?
* The level/spread split from `scoring.evaluate`: the level change should be zero by construction.
* The validation curves: are 30 epochs enough, and does early stopping fire at a sensible point?
* The attention weights on a few reports: are they spread evenly or concentrated on a few chunks, and for the
  "both" variant, which section gets the weight?
* The run log: seconds per epoch against the number of training reports, for the scaling section.

## References

- Hastie, T., Tibshirani, R. & Friedman, J. (2009). *The Elements of Statistical Learning*, 2nd ed., §11.5 (training neural networks). Springer.
- Ilse, M., Tomczak, J. M. & Welling, M. (2018). Attention-based deep multiple instance learning. *ICML 2018*.
- Kingma, D. P. & Ba, J. (2015). Adam: A method for stochastic optimization. *ICLR 2015*.
- Loshchilov, I. & Hutter, F. (2019). Decoupled weight decay regularization. *ICLR 2019*.
- Srivastava, N., Hinton, G., Krizhevsky, A., Sutskever, I. & Salakhutdinov, R. (2014). Dropout: A simple way to prevent neural networks from overfitting. *Journal of Machine Learning Research*, 15, 1929–1958.
- Xia, B., Rawte, V. D., Zaki, M. J. & Gupta, A. (2022). FETILDA: An effective framework for fin-tuned embeddings for long financial text documents. *arXiv:2206.06952*.

Course material (Data Science Toolbox, University of Bristol, 2026):

- Lecture 02.2, Cross Validation. https://dsbristol.github.io/dst/assets/slides/02.2-CrossValidation.pdf
- Lecture 05.2, Towards Modern Statistical Testing. https://dsbristol.github.io/dst/assets/slides/05.2-StatisticalTesting.pdf
- Lecture 06.1, Neural Networks Lecture 1. https://dsbristol.github.io/dst/assets/slides/06.1-Neural%20Networks%20Lecture%201.pdf
- Lecture 06.2, Neural Networks Lecture 2 (CNNs and Transformers). https://dsbristol.github.io/dst/assets/slides/06.2-Neural%20Networks%20Lecture%202.pdf
- Workshop 06.3, PyTorch for Computer Vision. https://github.com/dsbristol/dst/blob/master/assets/workshops/block06-DSToolboxWeek7NNLab2026.ipynb
- Lecture 09.1, Introduction to Parallelism. https://dsbristol.github.io/dst/assets/slides/09.1-ParallelAlgorithms.pdf
- Lecture 09.2, Mapping, Reducing, and Computational Graphs. https://dsbristol.github.io/dst/assets/slides/09.2-ComputationalGraphs.pdf